# Intro to Databases for Data Engineering

This notebook introduces databases and SQL using **Pandas + SQLite in Google Colab**.

No local database installation is required.

SQLite is built into Python through the `sqlite3` module.

Topics covered:
- What is a database?
- Tables, rows, columns, and schema
- Primary keys and foreign keys
- Loading Pandas DataFrames into SQLite tables
- Running SQL using `pd.read_sql_query()`
- SELECT, WHERE, ORDER BY, LIMIT
- Aggregations and GROUP BY
- INNER JOIN and LEFT JOIN
- Right join logic by swapping tables
- Full outer join simulation in SQLite
- Data quality checks using SQL
- Views and analytics tables
- OLTP vs OLAP
- Mini lab
- Interview questions


## 1. Why Databases Matter in Data Engineering

Data Engineers use databases to:
- store raw and cleaned data
- query structured datasets
- join related tables
- validate pipeline outputs
- create summary tables for analytics
- serve dashboards and reporting systems

A common Data Engineering flow:

```text
Source System
    ↓
Raw Table
    ↓
Cleaned Table
    ↓
Analytics Table
    ↓
Dashboard / ML / Reporting
```


## 2. Import Libraries

SQLite is available by default in Python.

No installation is required.


In [ ]:
import pandas as pd
import sqlite3
from IPython.display import display

print("Pandas version:", pd.__version__)
print("SQLite is ready")


## 3. Database Concepts

A database stores structured data.

Common terms:
- Database: collection of related tables
- Table: structured data in rows and columns
- Row: one record
- Column: one attribute
- Schema: structure of a table
- Primary key: uniquely identifies a row
- Foreign key: connects one table to another


## 4. Create Sample E-Commerce Tables

These Pandas DataFrames will become database tables in SQLite.


In [ ]:
customers = pd.DataFrame({
    "customer_id": [501, 502, 503, 504, 505],
    "customer_name": ["Riya", "Aarav", "Kabir", "Meera", "Isha"],
    "city": ["Delhi", "Mumbai", "Bangalore", "Pune", "Chennai"],
    "segment": ["Premium", "Standard", "Standard", "Premium", "Standard"]
})

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006],
    "customer_id": [501, 502, 501, 503, 999, 504],
    "order_date": ["2026-01-01", "2026-01-02", "2026-01-03", "2026-01-04", "2026-01-05", "2026-01-06"],
    "amount": [2500, 1800, 3200, 900, 700, 5000],
    "status": ["completed", "completed", "pending", "completed", "completed", "cancelled"]
})

products = pd.DataFrame({
    "product_id": [201, 202, 203, 204],
    "product_name": ["Laptop", "Phone", "Shoes", "Keyboard"],
    "category": ["Electronics", "Electronics", "Fashion", "Electronics"],
    "price": [80000, 40000, 3000, 1500]
})

order_items = pd.DataFrame({
    "order_id": [1001, 1001, 1002, 1003, 1004, 1006, 1006],
    "product_id": [201, 204, 202, 203, 203, 201, 202],
    "quantity": [1, 2, 1, 3, 1, 1, 1]
})

payments = pd.DataFrame({
    "payment_id": [9001, 9002, 9003, 9004, 9005],
    "order_id": [1001, 1002, 1003, 1004, 1006],
    "payment_status": ["success", "success", "pending", "success", "failed"],
    "payment_method": ["card", "upi", "card", "wallet", "card"]
})

print("customers")
display(customers)

print("orders")
display(orders)

print("products")
display(products)

print("order_items")
display(order_items)

print("payments")
display(payments)


## 5. Create an In-Memory SQLite Database

`sqlite3.connect(':memory:')` creates a temporary database in memory.

This is perfect for Google Colab practice.

The database disappears when the runtime stops.


In [ ]:
conn = sqlite3.connect(":memory:")

print("SQLite in-memory database created")


## 6. Load Pandas DataFrames into SQLite Tables

`to_sql()` writes a Pandas DataFrame into a SQL table.


In [ ]:
customers.to_sql("customers", conn, index=False, if_exists="replace")
orders.to_sql("orders", conn, index=False, if_exists="replace")
products.to_sql("products", conn, index=False, if_exists="replace")
order_items.to_sql("order_items", conn, index=False, if_exists="replace")
payments.to_sql("payments", conn, index=False, if_exists="replace")

print("Tables loaded into SQLite")


## 7. List Tables in SQLite

SQLite stores table information in `sqlite_master`.


In [ ]:
pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""", conn)


## 8. Inspect Table Schema

`PRAGMA table_info(table_name)` shows column metadata.


In [ ]:
pd.read_sql_query("PRAGMA table_info(orders)", conn)


## 9. Primary Key and Foreign Key Concepts

Logical primary key examples:
- customers.customer_id
- orders.order_id
- products.product_id
- payments.payment_id

Logical foreign key examples:
- orders.customer_id connects to customers.customer_id
- order_items.order_id connects to orders.order_id
- order_items.product_id connects to products.product_id
- payments.order_id connects to orders.order_id

In this notebook, these relationships are used for SQL joins and data quality checks.


# SQL Basics


## 10. SELECT All Rows


In [ ]:
pd.read_sql_query("""
SELECT *
FROM orders
""", conn)


## 11. SELECT Specific Columns


In [ ]:
pd.read_sql_query("""
SELECT
    order_id,
    customer_id,
    amount,
    status
FROM orders
""", conn)


## 12. WHERE Filter


In [ ]:
pd.read_sql_query("""
SELECT *
FROM orders
WHERE amount > 1000
""", conn)


## 13. Multiple Conditions


In [ ]:
pd.read_sql_query("""
SELECT *
FROM orders
WHERE amount > 1000
  AND status = 'completed'
""", conn)


## 14. ORDER BY


In [ ]:
pd.read_sql_query("""
SELECT *
FROM orders
ORDER BY amount DESC
""", conn)


## 15. LIMIT


In [ ]:
pd.read_sql_query("""
SELECT *
FROM orders
ORDER BY amount DESC
LIMIT 3
""", conn)


## 16. Derived Columns in SQL


In [ ]:
pd.read_sql_query("""
SELECT
    order_id,
    customer_id,
    amount,
    amount * 0.18 AS tax,
    amount + (amount * 0.18) AS final_amount
FROM orders
WHERE status = 'completed'
""", conn)


# Aggregations


## 17. Total Revenue


In [ ]:
pd.read_sql_query("""
SELECT
    SUM(amount) AS total_revenue
FROM orders
WHERE status = 'completed'
""", conn)


## 18. Count Orders by Status


In [ ]:
pd.read_sql_query("""
SELECT
    status,
    COUNT(*) AS order_count
FROM orders
GROUP BY status
ORDER BY order_count DESC
""", conn)


## 19. Revenue by Customer ID


In [ ]:
pd.read_sql_query("""
SELECT
    customer_id,
    COUNT(*) AS total_orders,
    SUM(amount) AS total_amount,
    AVG(amount) AS average_order_value,
    MIN(amount) AS min_order_value,
    MAX(amount) AS max_order_value
FROM orders
GROUP BY customer_id
ORDER BY total_amount DESC
""", conn)


## 20. HAVING Clause

`WHERE` filters rows before grouping.

`HAVING` filters groups after aggregation.


In [ ]:
pd.read_sql_query("""
SELECT
    customer_id,
    SUM(amount) AS total_amount
FROM orders
GROUP BY customer_id
HAVING SUM(amount) > 2000
ORDER BY total_amount DESC
""", conn)


## 21. SQL GROUP BY and Pandas groupby

SQL:

```sql
SELECT customer_id, SUM(amount)
FROM orders
GROUP BY customer_id
```

Pandas equivalent:

```python
orders.groupby("customer_id")["amount"].sum()
```


In [ ]:
orders.groupby("customer_id")["amount"].sum()


# Joins


## 22. INNER JOIN

INNER JOIN returns only matching records from both tables.


In [ ]:
pd.read_sql_query("""
SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    c.city,
    o.amount,
    o.status
FROM orders o
INNER JOIN customers c
    ON o.customer_id = c.customer_id
ORDER BY o.order_id
""", conn)


## 23. LEFT JOIN

LEFT JOIN keeps all records from the left table.

If there is no match in the right table, the right-side columns become NULL.


In [ ]:
pd.read_sql_query("""
SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    c.city,
    o.amount,
    o.status
FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
ORDER BY o.order_id
""", conn)


## 24. Right Join Logic in SQLite

SQLite practice is often simpler with LEFT JOIN.

To think like a RIGHT JOIN, swap table positions and use LEFT JOIN.

This keeps all customers and matching orders.


In [ ]:
pd.read_sql_query("""
SELECT
    c.customer_id,
    c.customer_name,
    c.city,
    o.order_id,
    o.amount,
    o.status
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
ORDER BY c.customer_id, o.order_id
""", conn)


## 25. Full Outer Join Simulation in SQLite

A full outer join keeps all records from both tables.

SQLite compatibility can vary, so this notebook simulates it using:
- left join from orders to customers
- union with customer records that have no matching orders


In [ ]:
pd.read_sql_query("""
SELECT
    o.order_id,
    o.customer_id AS order_customer_id,
    c.customer_id AS customer_table_id,
    c.customer_name,
    c.city,
    o.amount,
    o.status
FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id

UNION ALL

SELECT
    o.order_id,
    o.customer_id AS order_customer_id,
    c.customer_id AS customer_table_id,
    c.customer_name,
    c.city,
    o.amount,
    o.status
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
""", conn)


# Data Quality Checks Using SQL


## 26. Invalid Foreign Keys

Find orders where `customer_id` does not exist in the customers table.


In [ ]:
pd.read_sql_query("""
SELECT
    o.order_id,
    o.customer_id,
    o.amount,
    o.status
FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL
""", conn)


## 27. Customers With No Orders


In [ ]:
pd.read_sql_query("""
SELECT
    c.customer_id,
    c.customer_name,
    c.city
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
""", conn)


## 28. Orders Without Successful Payment


In [ ]:
pd.read_sql_query("""
SELECT
    o.order_id,
    o.customer_id,
    o.amount,
    o.status,
    p.payment_status
FROM orders o
LEFT JOIN payments p
    ON o.order_id = p.order_id
WHERE p.payment_status IS NULL
   OR p.payment_status != 'success'
ORDER BY o.order_id
""", conn)


# Multi-Table Joins


## 29. Join Orders, Order Items, and Products

This builds product-level order details.


In [ ]:
pd.read_sql_query("""
SELECT
    oi.order_id,
    oi.product_id,
    p.product_name,
    p.category,
    oi.quantity,
    p.price,
    oi.quantity * p.price AS line_total
FROM order_items oi
INNER JOIN products p
    ON oi.product_id = p.product_id
ORDER BY oi.order_id
""", conn)


## 30. Product Revenue by Category


In [ ]:
pd.read_sql_query("""
SELECT
    p.category,
    SUM(oi.quantity * p.price) AS category_revenue,
    SUM(oi.quantity) AS units_sold
FROM order_items oi
INNER JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY category_revenue DESC
""", conn)


## 31. Customer Paid Revenue Report

This joins:
- customers
- orders
- payments

It calculates successful paid revenue by customer.


In [ ]:
pd.read_sql_query("""
SELECT
    c.customer_id,
    c.customer_name,
    c.city,
    COUNT(o.order_id) AS total_orders,
    SUM(o.amount) AS paid_revenue
FROM customers c
INNER JOIN orders o
    ON c.customer_id = o.customer_id
INNER JOIN payments p
    ON o.order_id = p.order_id
WHERE p.payment_status = 'success'
GROUP BY
    c.customer_id,
    c.customer_name,
    c.city
ORDER BY paid_revenue DESC
""", conn)


# Views and Analytics Tables


## 32. Create a Clean Orders View

A view stores a reusable SQL query.

Here, we create a clean completed orders view.


In [ ]:
conn.executescript("""
DROP VIEW IF EXISTS clean_completed_orders;

CREATE VIEW clean_completed_orders AS
SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    c.city,
    o.order_date,
    o.amount,
    o.status
FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
  AND o.amount > 0
  AND c.customer_id IS NOT NULL;
""")

print("View created")


In [ ]:
pd.read_sql_query("""
SELECT *
FROM clean_completed_orders
""", conn)


## 33. Create an Analytics Summary Table

This simulates creating a reporting table for dashboards.


In [ ]:
city_revenue_summary = pd.read_sql_query("""
SELECT
    city,
    COUNT(*) AS completed_orders,
    SUM(amount) AS total_revenue,
    AVG(amount) AS average_order_value
FROM clean_completed_orders
GROUP BY city
ORDER BY total_revenue DESC
""", conn)

city_revenue_summary


## 34. Write SQL Result Back to SQLite

A query result can be written back as a new table.


In [ ]:
city_revenue_summary.to_sql("city_revenue_summary", conn, index=False, if_exists="replace")

pd.read_sql_query("SELECT * FROM city_revenue_summary", conn)


# Working with Files and SQLite


## 35. Save Source Data to CSV


In [ ]:
orders.to_csv("orders.csv", index=False)
customers.to_csv("customers.csv", index=False)

print("orders.csv and customers.csv created")


## 36. Load CSV into Pandas, Then Into SQLite

This is a common no-setup Data Engineering pattern in Colab.


In [ ]:
orders_from_csv = pd.read_csv("orders.csv")
orders_from_csv.to_sql("orders_from_csv", conn, index=False, if_exists="replace")

pd.read_sql_query("""
SELECT *
FROM orders_from_csv
WHERE amount > 1000
""", conn)


# OLTP vs OLAP


## 37. OLTP and OLAP

OLTP:
- Online Transaction Processing
- handles application transactions
- examples: order placement, payments, account updates
- databases: PostgreSQL, MySQL, SQL Server

OLAP:
- Online Analytical Processing
- handles analytics and reporting
- examples: dashboards, revenue summaries, customer segmentation
- platforms: BigQuery, Snowflake, Redshift, Databricks, DuckDB

Data Engineering often moves data from OLTP systems to OLAP systems.


## 38. Data Engineering Database Flow

A common database flow:

```text
Application Database
    ↓
Raw Extract
    ↓
Staging Table
    ↓
Cleaned Table
    ↓
Aggregated Analytics Table
    ↓
Dashboard
```

Example tables:
- orders_raw
- orders_clean
- daily_sales_summary


# Mini Lab


## 39. Mini Lab: E-Commerce Revenue Analysis

Use SQLite SQL queries to answer the tasks below.


### Task 1: Show all orders


In [ ]:
# Write SQL here using pd.read_sql_query()


### Task 2: Show only completed orders


In [ ]:
# Write SQL here using pd.read_sql_query()


### Task 3: Find total revenue from completed orders


In [ ]:
# Write SQL here using pd.read_sql_query()


### Task 4: Count orders by status


In [ ]:
# Write SQL here using pd.read_sql_query()


### Task 5: Join customers with orders


In [ ]:
# Write SQL here using pd.read_sql_query()


### Task 6: Find revenue by city


In [ ]:
# Write SQL here using pd.read_sql_query()


### Task 7: Find customers with no orders


In [ ]:
# Write SQL here using pd.read_sql_query()


### Task 8: Identify orders with invalid customer IDs


In [ ]:
# Write SQL here using pd.read_sql_query()


### Task 9: Find revenue by product category


In [ ]:
# Write SQL here using pd.read_sql_query()


### Task 10: Find orders without successful payment


In [ ]:
# Write SQL here using pd.read_sql_query()


# Interview Questions


## 40. What is the difference between a database and a table?

A database is a collection of related tables.

A table stores structured data in rows and columns.


## 41. What is a primary key?

A primary key uniquely identifies each row in a table.

Examples:
- customers.customer_id
- orders.order_id
- products.product_id


## 42. What is a foreign key?

A foreign key connects one table to another.

Example:
- orders.customer_id refers to customers.customer_id


## 43. What is the difference between INNER JOIN and LEFT JOIN?

INNER JOIN:
- returns only matching records from both tables

LEFT JOIN:
- returns all records from the left table
- returns matching records from the right table
- fills unmatched right-side columns with NULL


## 44. How do you perform a LEFT JOIN in SQL?

```sql
SELECT *
FROM left_table l
LEFT JOIN right_table r
    ON l.key = r.key;
```


## 45. Why do Data Engineers use SQL?

Data Engineers use SQL to:
- query data
- filter records
- join tables
- aggregate data
- validate pipeline outputs
- create analytics tables
- debug data quality issues


## 46. What is the difference between OLTP and OLAP?

OLTP:
- optimized for transactions
- used by applications
- handles many small writes and reads

OLAP:
- optimized for analytics
- used by dashboards and reporting
- handles large scans and aggregations


## 47. Summary

Key takeaways:
- SQLite can be used in Colab without local setup.
- Pandas DataFrames can be loaded into SQLite tables using `to_sql()`.
- SQL results can be read back into Pandas using `pd.read_sql_query()`.
- Primary keys identify records.
- Foreign keys connect tables.
- SQL is used to filter, join, aggregate, validate, and summarize data.
- Data Engineers use SQL heavily in pipelines, warehouses, and analytics systems.
